# Patram-7B GPU backend (Colab / Kaggle)

Runs **BharatGen Patram-7B-Instruct** behind an OpenAI-compatible API and exposes it through a free Cloudflare tunnel, so the Gradio app on your laptop can use it (`PATRAM_API_BASE`).
You can also run the full evaluation with Patram here.

**GPU:** T4 16 GB → 4-bit (`PATRAM_4BIT=1`). L4 / A100 → bf16. Kaggle 2×T4 → fp16 with `device_map=auto`.

**Before you start:** accept the terms on https://huggingface.co/bharatgenai/patram-7b-instruct and the two IndicTrans2 pages, and add your HF token as a Colab secret named `HF_TOKEN`.

In [ ]:
!nvidia-smi

In [ ]:
REPO = "https://github.com/rishabh-a-v/indic-doc-understanding.git"
!git clone -q $REPO repo
%cd repo
!pip -q install -r requirements-gpu.txt

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!huggingface-cli login --token $HF_TOKEN

In [ ]:
import subprocess, time, requests
os.environ["PATRAM_4BIT"] = "1"
server = subprocess.Popen(["python", "patram_server.py"], stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok: print("server up"); break
    except Exception: pass
    time.sleep(10)
!tail -5 server.log

In [ ]:
import base64, requests
img = base64.b64encode(open("eval/testset/00_hi_application_form.png","rb").read()).decode()
r = requests.post("http://localhost:8000/v1/chat/completions", json={"messages":[{"role":"user","content":[
    {"type":"image_url","image_url":{"url":"data:image/png;base64,"+img}},
    {"type":"text","text":"Extract all key-value pairs from this form as JSON."}]}], "max_tokens":300})
print(r.json()["choices"][0]["message"]["content"])

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=open("tunnel.log","w"), stderr=subprocess.STDOUT)
time.sleep(8)
!grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.log | head -1

### Optional: run the whole evaluation with Patram on this GPU

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-{hin,mar,ben,tam,tel,kan,guj,mal,pan,ori} > /dev/null
!pip -q install IndicTransToolkit sentence-transformers pytesseract pymupdf opencv-python-headless rapidfuzz gradio pandas matplotlib sacremoses
os.environ["PATRAM_API_BASE"] = "http://localhost:8000/v1"
!python -m eval.run_eval --patram